# LH-DNN Current-Run Trade-Off Plots

LH-DNN contributes a single arm: the paper-derived large-topology baseline
(`lhdnn_<dataset>`). Native lexicographic training is not supported for LH-DNN in
this repository, and there is
no HCC launcher for it, so there is no within-family run matrix to compare. The
figures below therefore locate LH-DNN among the other model families rather than
isolating a mechanism inside it.

Two scope notes carry into every reading of these numbers:

- LH-DNN's own hierarchy mechanism is the branch-point projection
  `z - c + sg(c)`, which leaves the forward value unchanged and only edits the
  backward pass. It is a gradient-space method and is inactive at inference, so
  it does not enforce consistency at decode time; the independent TICE column is
  the honest measure of what the trained network produces.
- The CIFAR-100 preset is paper-aligned (large topology, 15-epoch schedule). The
  CUB-200 and Aircraft presets are explicit local extrapolations that keep the
  topology but reduce the final feature map with deterministic 7x7 average
  pooling. Cross-dataset statements about LH-DNN should say which of the two
  they rest on.

Hier-COS runs that borrow the LH-DNN projection layer are a separate study and
live in `hiercos_current_plots.ipynb`, not here.

FPA, weighted AP, and level accuracy are higher-is-better; TICE and AHD are
lower-is-better. Points, bars, and error bars are mean +/- sample standard
deviation across training seeds. A trailing `*` marks a run whose
`test_metrics.yaml` predates the top-down/independent selection split, so its
checkpoint was chosen by a single criterion.

### Selection and metric family

Every figure and table below reads the **independently selected** checkpoint and
the independent metric family. Top-down decoding is deliberately not offered:
its predicted path is consistent by construction, so `tice_topdown` is exactly
`0.0` in all 194 completed seed directories under `/scratch/g.saggini1/outputs`
(checked 2026-08-05) and `fpa_topdown` equals top-down fine accuracy exactly,
which leaves a top-down trade-off view with nothing to show.

### Figure format

Figures are authored at their final printed size (6.3 in, an A4 text block with
2.5 cm margins) in the same style as `analysis/datasets_analysis.ipynb`, and are
written to `FIGURE_DIR` as PDF and PNG. Include them with `width=\linewidth`
and no further scaling: any resizing in LaTeX shrinks the fonts along with the
artwork. Figure titles are omitted on purpose — they belong in the caption.

One dataset per row rather than three panels side by side: at 6.3 in a 1x3 row
leaves about 2 in per panel, which is not enough for the direct labels or for
grouped bars once a run matrix grows.

In [ ]:
from pathlib import Path
import sys

# The helper lives in notebooks/utils; also look upward so the notebook works
# when the kernel starts at the repository root.
SEARCH_PATHS = [Path.cwd(), *(parent / 'notebooks' / 'utils' for parent in (Path.cwd(), *Path.cwd().parents))]
HELPER_DIR = next((path for path in SEARCH_PATHS if (path / 'current_run_plot_utils.py').is_file()), None)
if HELPER_DIR is None:
    raise FileNotFoundError('Run this notebook from the repository tree so notebooks/utils is discoverable.')
sys.path.insert(0, str(HELPER_DIR))

import matplotlib.pyplot as plt

from current_run_plot_utils import (
    MECHANISM_COLORS,
    VARIANT_MARKERS,
    check_encoding,
    discover_rows,
    encode_rows,
    model_reference_specs,
    plot_level_accuracy,
    plot_level_accuracy_deltas,
    plot_tradeoff,
    print_availability,
    print_reference_availability,
    print_summary,
    use_paper_style,
)

use_paper_style()

OUTPUTS_ROOT = Path('/scratch/g.saggini1/outputs')
# Each figure is written as a PDF and a PNG at its authored printed size.
SAVE_FIGURES = True
FIGURE_DIR = OUTPUTS_ROOT / 'analysis' / 'current_runs' / 'lhdnn'

DATASETS = {
    'cifar100': 'CIFAR-100',
    'cub200': 'CUB-200',
    'aircraft': 'Aircraft',
}

In [ ]:
RUN_SPECS = [
    {'key': 'baseline', 'label': 'LH-DNN baseline',
     'mechanism': 'baseline', 'variant': 'native', 'canonical': True,
     'run_name': 'lhdnn_{dataset}'},
]

# LH-DNN itself is the focal model, so it is excluded from the reference set.
REFERENCE_SPECS = model_reference_specs(exclude={'lhdnn'})

In [ ]:
rows, missing_runs = discover_rows(OUTPUTS_ROOT, DATASETS, RUN_SPECS)
reference_rows, missing_references = discover_rows(
    OUTPUTS_ROOT, DATASETS, REFERENCE_SPECS, pareto=False
)

# Colour encodes the mechanism and means the same thing in every model's
# notebook; shape separates the variants within a mechanism.
ENCODING = encode_rows(
    rows,
    hue=('mechanism', MECHANISM_COLORS),
    shape=('variant', VARIANT_MARKERS),
)

print_availability(rows, missing_runs, DATASETS)
print_reference_availability(reference_rows, missing_references, DATASETS)
check_encoding(rows, reference_rows, DATASETS)

## FPA Trade-Off

Each panel shows the individual seeds, the across-seed mean, a
one-standard-deviation covariance ellipse when at least three seeds are
available, and a thin crosshair through the canonical baseline so every other
point can be read as above/below and left/right of it.

All models share one axis range here, because the LH-DNN family contributes a
single arm and freezing the scale on one point would push every other model
off-scale. Reference models were trained with their own recipes and epoch
budgets, so this figure locates LH-DNN within the achievable operating region
rather than making a controlled comparison.

### Point encoding

Colour encodes the **mechanism** — the intervention a run applies — and means
the same thing in every model's notebook, so a colour learned here transfers to
`hcast_current_plots.ipynb` and the rest. Cross-model references give up colour
entirely and are drawn in neutral grey, told apart by marker shape, so the focal
family owns the colour dimension.

Because every model here sits on one shared scale, no marker is pinned into an
off-scale gutter, and the default `point_labels='off_scale'` therefore draws no
direct labels at all: the legend identifies every point and the axes carry the
values. Exact numbers with their standard deviations are in the availability and
summary tables. Pass `point_labels='auto'` to label the in-range references
directly instead.

In [ ]:
plot_tradeoff(
    rows, DATASETS, FIGURE_DIR, SAVE_FIGURES,
    reference_rows=reference_rows, reference_specs=REFERENCE_SPECS,
    encoding=ENCODING,
)

## Accuracy Per Hierarchy Level

Absolute coarse, middle, and fine accuracy for each run of this model family.

Cross-model references are deliberately left out. They were trained with their
own recipes and epoch budgets, so a bar beside them invites a controlled reading
the data does not support; the trade-off figure above already places them, and
the availability and summary tables carry their exact numbers. Dropping them
also keeps the bars wide enough to stay readable as the run matrix grows.

With a single LH-DNN arm this is the main quantitative view in the notebook: it
shows where the accuracy sits at each level rather than only the aggregate FPA.

In [ ]:
plot_level_accuracy(
    rows, DATASETS, FIGURE_DIR, SAVE_FIGURES,
)

## Selected-Run Effects Across Hierarchy Levels

The matched-seed delta figure needs at least one non-baseline arm sharing seeds
with the baseline. With only the native LH-DNN run available it reports that no
pair exists and draws nothing. The call is kept so the figure appears
automatically if a second LH-DNN arm is ever added.

In [ ]:
plot_level_accuracy_deltas(
    rows, DATASETS, FIGURE_DIR, SAVE_FIGURES,
    baseline_name='the native LH-DNN baseline',
)

## Run and Pareto Summary

A run is Pareto-optimal within a dataset when no other run of the same model
family has both higher-or-equal FPA and lower-or-equal TICE, with at least one
strict improvement. Reference models are not part of this comparison. AHD and
weighted AP are reported as additional columns but do not enter the Pareto test.

With one arm per dataset the Pareto column is trivially satisfied; the table is
useful here for the per-seed spread and the selected epoch. Compare the absolute
numbers against the reference-model table printed in the availability cell.

In [ ]:
print_summary(rows, DATASETS)